# SIP Future Value (Monthly Investment)

Let:

- $P$ = SIP amount invested each month  
- $R$ = expected annual return (in **decimal** form, e.g., 12% $\rightarrow$ 0.12)  
- $n$ = investment duration in years  
- $m = 12n$ = total number of months  
- $i$ = effective monthly return

### Convert annual return to monthly return

$$
i = (1 + R)^{\frac{1}{12}} - 1
$$

### Total amount after $n$ years (Future Value)

#### 1) SIP invested at the **end of each month** (ordinary annuity)

$$
FV = P \times \frac{(1+i)^{m} - 1}{i}
$$

#### 2) SIP invested at the **start of each month** (annuity due)

$$
FV = P \times \frac{(1+i)^{m} - 1}{i} \times (1+i)
$$

### Total invested and gains

Total invested:

$$
\text{Invested} = P \times m
$$

Estimated gain:

$$
\text{Gain} = FV - (P \times m)
$$


In [ ]:
def sip_future_value(P, R, n, annuity_due=False):
    """Calculate the future value of a Systematic Investment Plan (SIP).
    
    Arguments:
        P (float): SIP amount invested each month
        R (float): Expected annual return in % form (12%)
        n (int): Investment duration in years
        annuity_due (bool): True if SIP is invested at the start of each month, False for end of month
    
    Returns:
        dict: Dictionary containing FV, invested, and gain
    """
    R = R / 100  # convert percentage to decimal
    m = 12 * n  # total number of months
    i = (1 + R)**(1/12) - 1  # effective monthly return
    
    # Future Value calculation
    fv = P * ((1 + i)**m - 1) / i
    if annuity_due:
        fv *= (1 + i)
    
    invested = P * m
    gain = fv - invested
    
    return {
        'future_value': fv,
        'invested': invested,
        'gain': gain
    }


In [ ]:
result = sip_future_value(P=1500, R=12, n=10, annuity_due=True) # Investing at the start of each month

print(f"Future Value: {result['future_value']:.2f}")
print(f"Invested: {result['invested']:.2f}")
print(f"Gain: {result['gain']:.2f}")

In [ ]:
# import sys, site, subprocess

# print("Python executable used by THIS notebook kernel:\n", sys.executable)
# print("\nSite-packages:\n", "\n".join(site.getsitepackages()))

# # Install into THIS exact kernel interpreter
# subprocess.check_call([sys.executable, "-m", "pip", "install", "-U", "nbformat>=4.2.0", "ipython", "plotly"])
# print("\nDone. Now: Kernel -> Restart (required once), then run Cell 2.")


## Single Mutual Fund (Invested VS Est. returns)

In [ ]:
# Reliable in-cell Plotly display in VSCode Jupyter (no hosting, no browser)
import sys, subprocess

# Ensure display stack + widgets are present in THIS kernel
subprocess.check_call([sys.executable, "-m", "pip", "install", "-U",
                       "ipython", "nbformat>=4.2.0", "ipywidgets", "plotly"])

import plotly.graph_objects as go
from IPython.display import display, HTML

# ---- SIP parameters ----
P = 1500
R_percent = 12
years_total = 10

R = R_percent / 100
m = int(12 * years_total)
i = (1 + R) ** (1/12) - 1
annuity_due = True

months = list(range(1, m + 1))
invested_cum = [P * month for month in months]

fv_cum = []
for month in months:
    if abs(i) < 1e-12:
        fv = P * month
    else:
        fv = P * (((1 + i) ** month - 1) / i)
        if annuity_due:
            fv *= (1 + i)
    fv_cum.append(fv)

x_years = [month / 12 for month in months]

# ---- Line plot ----
fig = go.Figure()
fig.add_trace(go.Scatter(x=x_years, y=invested_cum, mode="lines", name="Invested Amount"))
fig.add_trace(go.Scatter(x=x_years, y=fv_cum, mode="lines", name="Future Value (Invested + Gain)"))
fig.update_layout(title="SIP Growth Over Time", xaxis_title="Years", yaxis_title="Amount (INR)")

# Display inside the cell as HTML (plotly.js embedded inline, no internet needed)
display(HTML(fig.to_html(include_plotlyjs=True, full_html=False)))

# ---- Donut chart ----
final_invested = invested_cum[-1]
final_fv = fv_cum[-1]
final_gain = final_fv - final_invested

fig2 = go.Figure(data=[go.Pie(labels=["Invested Amount", "Estimated Gain"],
                              values=[final_invested, final_gain],
                              hole=0.3)])
fig2.update_layout(title="Invested vs Estimated Returns")

display(HTML(fig2.to_html(include_plotlyjs=True, full_html=False)))


## 3 Mutual Funds Invested vs Total Returns

In [ ]:
# Portfolio SIP (3 MFs) + combined charts + Indian ₹ formatting (Indian commas + Cr/Lakh + words)
# Works in VSCode Jupyter (renders inside the cell) using Plotly HTML embedding.

import math
import plotly.graph_objects as go
from IPython.display import display, HTML

# -----------------------------
# 1) Indian money formatting
# -----------------------------
RUPEE = "₹"

def indian_commas(num: float) -> str:
    """12,34,56,789 style commas (keeps 0 decimals by default)."""
    n = int(round(num))
    s = str(abs(n))
    if len(s) <= 3:
        out = s
    else:
        last3 = s[-3:]
        rest = s[:-3]
        parts = []
        while len(rest) > 2:
            parts.append(rest[-2:])
            rest = rest[:-2]
        if rest:
            parts.append(rest)
        out = ",".join(reversed(parts)) + "," + last3
    return ("-" if n < 0 else "") + out

def inr_compact(num: float) -> str:
    """Compact: ₹12.34Cr / ₹56.78L / ₹90.12K / ₹1,234"""
    n = float(num)
    sign = "-" if n < 0 else ""
    n = abs(n)
    crore = 10_000_000
    lakh  = 100_000
    thousand = 1_000

    if n >= crore:
        return f"{sign}{RUPEE}{n/crore:.2f}Cr"
    if n >= lakh:
        return f"{sign}{RUPEE}{n/lakh:.2f}L"
    if n >= thousand:
        return f"{sign}{RUPEE}{n/thousand:.2f}K"
    return f"{sign}{RUPEE}{indian_commas(n)}"

def inr_words(num: float) -> str:
    """
    Indian grouping words:
    e.g., 12,34,56,789 -> 12 crore 34 lakh 56 thousand 789
    (simple grouping, not full English spellout of 789)
    """
    n = int(round(num))
    sign = "minus " if n < 0 else ""
    n = abs(n)

    crore = n // 10_000_000
    n %= 10_000_000
    lakh = n // 100_000
    n %= 100_000
    thousand = n // 1000
    n %= 1000
    remainder = n

    parts = []
    if crore: parts.append(f"{crore} crore")
    if lakh: parts.append(f"{lakh} lakh")
    if thousand: parts.append(f"{thousand} thousand")
    if remainder or not parts: parts.append(f"{remainder}")
    return sign + " ".join(parts)

def fmt_rupee(num: float) -> str:
    return f"{RUPEE}{indian_commas(num)}"

# -----------------------------
# 2) SIP simulation (accurate month-by-month)
# -----------------------------
def monthly_rate_from_annual(R_percent: float) -> float:
    R = R_percent / 100.0
    return (1 + R) ** (1/12) - 1

def simulate_sip(
    P: float,
    R_percent: float,
    horizon_years: float,
    annuity_due: bool = True,
    sip_years: float | None = None,
):
    """
    Month-by-month simulation:
    - annuity_due=True: invest at START of month -> (balance + contrib) * (1+i)
    - annuity_due=False: invest at END of month -> balance*(1+i) + contrib
    - sip_years: if you want to stop SIP early; after that, balance just compounds.
    """
    i = monthly_rate_from_annual(R_percent)
    m = int(round(12 * horizon_years))
    sip_m = int(round(12 * (sip_years if sip_years is not None else horizon_years)))

    balance = 0.0
    invested_cum = []
    fv_cum = []
    invested = 0.0

    for month in range(1, m + 1):
        contrib = P if month <= sip_m else 0.0
        invested += contrib

        if annuity_due:
            balance = (balance + contrib) * (1 + i)
        else:
            balance = balance * (1 + i) + contrib

        invested_cum.append(invested)
        fv_cum.append(balance)

    return {
        "months": m,
        "monthly_rate": i,
        "invested_series": invested_cum,
        "fv_series": fv_cum,
        "invested_final": invested_cum[-1] if invested_cum else 0.0,
        "fv_final": fv_cum[-1] if fv_cum else 0.0,
        "gain_final": (fv_cum[-1] - invested_cum[-1]) if fv_cum else 0.0,
    }

# -----------------------------
# 3) Configure your 3 mutual funds here
# -----------------------------
horizon_years = 10
annuity_due = True  # True = SIP at start of month (common assumption). Set False for end-of-month.

mfs = [
    {"name": "MF-1", "sip": 1500, "annual_return_percent": 12.0},
    {"name": "MF-2", "sip": 2000, "annual_return_percent": 11.0},
    {"name": "MF-3", "sip": 3000, "annual_return_percent": 13.0},
]

# -----------------------------
# 4) Run simulations + portfolio totals
# -----------------------------
results = []
max_months = int(round(12 * horizon_years))

for mf in mfs:
    res = simulate_sip(
        P=mf["sip"],
        R_percent=mf["annual_return_percent"],
        horizon_years=horizon_years,
        annuity_due=annuity_due,
        sip_years=horizon_years,   # change if you want SIP to stop earlier for any MF
    )
    res["name"] = mf["name"]
    res["sip"] = mf["sip"]
    res["annual_return_percent"] = mf["annual_return_percent"]
    results.append(res)

portfolio_invested = [0.0] * max_months
portfolio_fv = [0.0] * max_months

for res in results:
    inv = res["invested_series"]
    fv = res["fv_series"]
    for idx in range(max_months):
        portfolio_invested[idx] += inv[idx]
        portfolio_fv[idx] += fv[idx]

portfolio_invested_final = portfolio_invested[-1]
portfolio_fv_final = portfolio_fv[-1]
portfolio_gain_final = portfolio_fv_final - portfolio_invested_final

x_years = [(k + 1) / 12 for k in range(max_months)]

# -----------------------------
# 5) Helper to display Plotly in-cell (no browser/hosting)
# -----------------------------
def show_in_cell(fig: go.Figure, height: int = 450):
    html = fig.to_html(include_plotlyjs=True, full_html=False, config={"displayModeBar": True})
    display(HTML(f"<div style='height:{height}px'>{html}</div>"))

# -----------------------------
# 6) Combined line plot (portfolio total)
# -----------------------------
title = (
    f"Portfolio SIP Growth (Total of {len(mfs)} MFs) — "
    f"Total: {fmt_rupee(portfolio_fv_final)} ({inr_compact(portfolio_fv_final)})"
)

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=x_years, y=portfolio_invested, mode="lines", name="Total Invested",
    customdata=[fmt_rupee(v) for v in portfolio_invested],
    hovertemplate="Years: %{x:.2f}<br>Total Invested: %{customdata}<extra></extra>"
))
fig.add_trace(go.Scatter(
    x=x_years, y=portfolio_fv, mode="lines", name="Total Future Value",
    customdata=[fmt_rupee(v) for v in portfolio_fv],
    hovertemplate="Years: %{x:.2f}<br>Total Future Value: %{customdata}<extra></extra>"
))
fig.update_layout(title=title, xaxis_title="Years", yaxis_title=f"Amount ({RUPEE})")
show_in_cell(fig, height=520)

# -----------------------------
# 7) Pie charts (3 pies): Invested by MF, Gains by MF, Total by MF
# -----------------------------
names = [r["name"] for r in results]
invested_vals = [r["invested_final"] for r in results]
gain_vals = [r["gain_final"] for r in results]
total_vals = [r["fv_final"] for r in results]

def pie_with_money_title(values, labels, title_text):
    text = [f"{fmt_rupee(v)} ({inr_compact(v)})" for v in values]
    figp = go.Figure(data=[
        go.Pie(
            labels=labels,
            values=values,
            hole=0.35,
            text=text,
            textinfo="label+percent",
            hovertemplate="%{label}<br>%{text}<extra></extra>"
        )
    ])
    figp.update_layout(title=title_text)
    return figp

fig_inv = pie_with_money_title(
    invested_vals, names,
    f"Invested Amount Split by MF — Total Invested: {fmt_rupee(portfolio_invested_final)} ({inr_compact(portfolio_invested_final)})"
)
show_in_cell(fig_inv, height=430)

fig_gain = pie_with_money_title(
    gain_vals, names,
    f"Gains Split by MF — Total Gain: {fmt_rupee(portfolio_gain_final)} ({inr_compact(portfolio_gain_final)})"
)
show_in_cell(fig_gain, height=430)

fig_tot = pie_with_money_title(
    total_vals, names,
    f"Total Future Value Split by MF — Portfolio Total: {fmt_rupee(portfolio_fv_final)} ({inr_compact(portfolio_fv_final)})"
)
show_in_cell(fig_tot, height=430)

# -----------------------------
# 8) OPTIONAL: per-MF donut (Invested vs Gain) for each MF
# -----------------------------
for r in results:
    inv = r["invested_final"]
    gain = r["gain_final"]
    tot = r["fv_final"]

    fig_mf = go.Figure(data=[
        go.Pie(
            labels=["Invested", "Gain"],
            values=[inv, gain],
            hole=0.35,
            text=[f"{fmt_rupee(inv)} ({inr_compact(inv)})", f"{fmt_rupee(gain)} ({inr_compact(gain)})"],
            textinfo="label+percent",
            hovertemplate="%{label}<br>%{text}<extra></extra>"
        )
    ])
    fig_mf.update_layout(
        title=(
            f"{r['name']} — SIP {fmt_rupee(r['sip'])}/month @ {r['annual_return_percent']}% p.a. | "
            f"Total: {fmt_rupee(tot)} ({inr_compact(tot)})"
        )
    )
    show_in_cell(fig_mf, height=420)

# -----------------------------
# 9) Quick text summary with Indian words
# -----------------------------
summary_html = f"""
<h3>Portfolio Summary</h3>
<ul>
  <li><b>Total Invested:</b> {fmt_rupee(portfolio_invested_final)} ({inr_compact(portfolio_invested_final)}) — {inr_words(portfolio_invested_final)}</li>
  <li><b>Total Gain:</b> {fmt_rupee(portfolio_gain_final)} ({inr_compact(portfolio_gain_final)}) — {inr_words(portfolio_gain_final)}</li>
  <li><b>Total Future Value:</b> {fmt_rupee(portfolio_fv_final)} ({inr_compact(portfolio_fv_final)}) — {inr_words(portfolio_fv_final)}</li>
</ul>
"""
display(HTML(summary_html))


## All 3 Mutual Funds in Single Plot as Subplots

In [ ]:
# Single figure with subplots:
# (1) Portfolio growth line chart
# (2) Invested split by MF (donut)
# (3) Gains split by MF (donut)
# (4) Total FV split by MF (donut)
# + optional per-MF donuts (Invested vs Gain) in extra rows if you want

import math
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display, HTML

# -----------------------------
# Indian money formatting
# -----------------------------
RUPEE = "₹"

def indian_commas(num: float) -> str:
    n = int(round(num))
    s = str(abs(n))
    if len(s) <= 3:
        out = s
    else:
        last3 = s[-3:]
        rest = s[:-3]
        parts = []
        while len(rest) > 2:
            parts.append(rest[-2:])
            rest = rest[:-2]
        if rest:
            parts.append(rest)
        out = ",".join(reversed(parts)) + "," + last3
    return ("-" if n < 0 else "") + out

def inr_compact(num: float) -> str:
    n = float(num)
    sign = "-" if n < 0 else ""
    n = abs(n)
    crore = 10_000_000
    lakh  = 100_000
    thousand = 1_000
    if n >= crore:
        return f"{sign}{RUPEE}{n/crore:.2f}Cr"
    if n >= lakh:
        return f"{sign}{RUPEE}{n/lakh:.2f}L"
    if n >= thousand:
        return f"{sign}{RUPEE}{n/thousand:.2f}K"
    return f"{sign}{RUPEE}{indian_commas(n)}"

def fmt_rupee(num: float) -> str:
    return f"{RUPEE}{indian_commas(num)}"

# -----------------------------
# SIP simulation
# -----------------------------
def monthly_rate_from_annual(R_percent: float) -> float:
    R = R_percent / 100.0
    return (1 + R) ** (1/12) - 1

def simulate_sip(P: float, R_percent: float, horizon_years: float, annuity_due: bool = True):
    i = monthly_rate_from_annual(R_percent)
    m = int(round(12 * horizon_years))
    balance = 0.0
    invested = 0.0
    invested_series, fv_series = [], []
    for month in range(1, m + 1):
        contrib = P
        invested += contrib
        if annuity_due:
            balance = (balance + contrib) * (1 + i)
        else:
            balance = balance * (1 + i) + contrib
        invested_series.append(invested)
        fv_series.append(balance)
    return {
        "months": m,
        "monthly_rate": i,
        "invested_series": invested_series,
        "fv_series": fv_series,
        "invested_final": invested_series[-1],
        "fv_final": fv_series[-1],
        "gain_final": fv_series[-1] - invested_series[-1],
    }

# -----------------------------
# Configure your 3 MFs
# -----------------------------
horizon_years = 10
annuity_due = True  # True = start-of-month SIP assumption

mfs = [
    {"name": "MF-1", "sip": 1500, "annual_return_percent": 12.0},
    {"name": "MF-2", "sip": 2000, "annual_return_percent": 11.0},
    {"name": "MF-3", "sip": 3000, "annual_return_percent": 13.0},
]

# -----------------------------
# Run simulations + portfolio totals
# -----------------------------
results = []
m = int(round(12 * horizon_years))

for mf in mfs:
    res = simulate_sip(mf["sip"], mf["annual_return_percent"], horizon_years, annuity_due)
    res["name"] = mf["name"]
    res["sip"] = mf["sip"]
    res["annual_return_percent"] = mf["annual_return_percent"]
    results.append(res)

portfolio_invested = [0.0] * m
portfolio_fv = [0.0] * m

for r in results:
    for idx in range(m):
        portfolio_invested[idx] += r["invested_series"][idx]
        portfolio_fv[idx] += r["fv_series"][idx]

portfolio_invested_final = portfolio_invested[-1]
portfolio_fv_final = portfolio_fv[-1]
portfolio_gain_final = portfolio_fv_final - portfolio_invested_final

x_years = [(k + 1) / 12 for k in range(m)]

names = [r["name"] for r in results]
invested_vals = [r["invested_final"] for r in results]
gain_vals = [r["gain_final"] for r in results]
total_vals = [r["fv_final"] for r in results]

# -----------------------------
# Create ONE figure with subplots
# -----------------------------
fig = make_subplots(
    rows=2, cols=2,
    specs=[
        [{"type": "xy"}, {"type": "domain"}],
        [{"type": "domain"}, {"type": "domain"}],
    ],
    subplot_titles=(
        f"Portfolio Growth (Total FV: {fmt_rupee(portfolio_fv_final)} / {inr_compact(portfolio_fv_final)})",
        f"Invested Split (Total: {fmt_rupee(portfolio_invested_final)})",
        f"Gains Split (Total: {fmt_rupee(portfolio_gain_final)})",
        f"Total FV Split (Total: {fmt_rupee(portfolio_fv_final)})",
    )
)

# (1) Line chart
fig.add_trace(
    go.Scatter(
        x=x_years, y=portfolio_invested, mode="lines", name="Total Invested",
        customdata=[fmt_rupee(v) for v in portfolio_invested],
        hovertemplate="Years: %{x:.2f}<br>Total Invested: %{customdata}<extra></extra>",
    ),
    row=1, col=1
)
fig.add_trace(
    go.Scatter(
        x=x_years, y=portfolio_fv, mode="lines", name="Total Future Value",
        customdata=[fmt_rupee(v) for v in portfolio_fv],
        hovertemplate="Years: %{x:.2f}<br>Total Future Value: %{customdata}<extra></extra>",
    ),
    row=1, col=1
)

fig.update_xaxes(title_text="Years", row=1, col=1)
fig.update_yaxes(title_text=f"Amount ({RUPEE})", row=1, col=1)

# (2) Invested split donut
fig.add_trace(
    go.Pie(
        labels=names, values=invested_vals, hole=0.4, name="Invested Split",
        text=[f"{fmt_rupee(v)} ({inr_compact(v)})" for v in invested_vals],
        textinfo="label+percent",
        hovertemplate="%{label}<br>%{text}<extra></extra>",
        showlegend=False
    ),
    row=1, col=2
)

# (3) Gains split donut
fig.add_trace(
    go.Pie(
        labels=names, values=gain_vals, hole=0.4, name="Gains Split",
        text=[f"{fmt_rupee(v)} ({inr_compact(v)})" for v in gain_vals],
        textinfo="label+percent",
        hovertemplate="%{label}<br>%{text}<extra></extra>",
        showlegend=False
    ),
    row=2, col=1
)

# (4) Total FV split donut
fig.add_trace(
    go.Pie(
        labels=names, values=total_vals, hole=0.4, name="Total Split",
        text=[f"{fmt_rupee(v)} ({inr_compact(v)})" for v in total_vals],
        textinfo="label+percent",
        hovertemplate="%{label}<br>%{text}<extra></extra>",
        showlegend=False
    ),
    row=2, col=2
)

fig.update_layout(
    title=f"3-MF SIP Portfolio Dashboard — Total: {fmt_rupee(portfolio_fv_final)} ({inr_compact(portfolio_fv_final)})",
    height=900,
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="left", x=0.0),
)

# Display in-cell (no browser/hosting)
display(HTML(fig.to_html(include_plotlyjs=True, full_html=False)))
